# SKEW固定信息增量
两会话滞后，原始未知不填补。控制为原64项价格/时点特征加SKEW可用性标志，候选仅加SKEW值。Ridge10、6次拟合；H5下行风险为主要目标，H5现金/杠杆收益为辅助门。全部市场行保留。来源历史vintage未认证，不构成晋升资格。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/skew_information_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.skew_information import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'skew_information_20261014',
 'plan_sha256': 'c5b485ea9447275cd74c4ac931f5e10875d7fd49107e88325539f6aa92e0b270',
 'decision': 'not_promoted_fixed_skew_information_failed',
 'information_gates': {'risk': False, 'returns': False},
 'objective_achieved': False,
 'new_fits': 6,
 'new_backtests': 0,
 'rows': 2006,
 'unknown_skew_rows': 3,
 'historical_vintage_certified': False,
 'normalization': 'same control training target scale for both methods',
 'recorder_id': 'd2cbede4024c4bf583304545e5cbd33b',
 'experiment_id': '766832790829256304',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/skew_information_20261014/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/skew_information_20261014/qlib_recorder/766832790829256304/d2cbede4024c4bf583304545e5cbd33b',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013',
 'region': 'us',
 'as_of': '2026-10-09',
 'qlib_docs': 'https://qlib.readthedocs.io/en/latest/

# SKEW固定信息增量：风险与收益门均失败

6次拟合，0组合回测。每个2021/2022/2023截止分别拟合控制与候选；Ridge alpha=10，主目标为H5观测下行风险，辅助目标为H5现金/杠杆相对收益。控制使用原64项价格/日内/时点特征加SKEW可用性标志，候选仅再加`(SKEW-100)/10`。辅助事件仅含双方相同的已知/年龄控制，不重新加入失败的指引数值。

两组使用同样的553/805/1056个成熟训练样本；最后标签分别在2020-12-31、2021-12-31、2022-12-30 09:49可用，严格早于拟合截止。训练均值只用于模型内部处理未知值，原始SKEW没有填补。信息误差统一按控制组训练标签尺度归一化，不使用候选尺度制造改善。

2006个市场输入日全部保留。SKEW在完整原生市场日历上滞后两会话，3个未知输入日为2018-12-06、2019-07-09、2024-12-03；2018-12-05为非交易日，不能按普通工作日补出。两组共有同一个缺失标志，未删除这些会话。SKEW假定最迟在来源下一会话17:00可用，第二会话09:46再消费；这是研究假设，未认证历史首发或修订版本。

下表为“控制误差减候选误差”的标准化MSE差，正数才是改善；不是收益百分点。

|窗口|目标组|改善|95%区块区间|
|---|---|---:|---:|
|development_2021|risk|0.01171|[-0.05208, 0.07799]|
|development_2021|returns|0.00895|[-0.04722, 0.06863]|
|development_2022|risk|-0.02332|[-0.03880, -0.00635]|
|development_2022|returns|-0.04972|[-0.09437, -0.00388]|
|reused_outer|risk|0.02730|[-0.03220, 0.09888]|
|reused_outer|returns|-0.02243|[-0.05368, 0.01251]|
|pooled_development|risk|-0.00577|[-0.04830, 0.04230]|
|pooled_development|returns|-0.02033|[-0.07141, 0.03394]|

主风险门要求两个开发年各自改善且开发期合并下界大于0；收益门对现金/杠杆联合误差施加相同要求。两门都失败。2022年风险与收益均恶化；2023年后风险改善区间跨零，不能用复用外窗的正点估计选择本候选。

开发检验有效标签分别246/245行，复用诊断940行，按父卡共同成熟掩码评分；未成熟输出仍保存在Recorder。主风险定义为未来5个持有区间收盘/退出09:47标记相对入场价的最大下跌，非完整分钟峰谷MaxDD。区块统计用固定63日平均块长、2000次stationary bootstrap，未校正所有历史研究选择。

决定：**not_promoted_fixed_skew_information_failed**。不启动组合回测、不反转SKEW、不改期限/alpha/阈值或缺失处理。此结果只否定本固定输入与模型合同，不宣称SKEW在所有市场或期限都无效。

稳定净SOXX超额目标未完成；生产SIM未改，源数据更新仍保留于NAS。当前研究尚未证明额外预测信息可支撑现金防守。下一机制必须有不同的可验证信息依据，并先核查来源时点与已有负面证据，不能从本轮外窗挑选有利参数。即使将来信息门通过，也仍需Tiered完整账户和独立前向资格。

[误差指标](forecast_metrics.csv)、[信息增量](information_increment.csv)、[成熟度](training_audit.csv)、[未知输入会话](unknown_sessions.csv)、[Recorder清单](verification_manifest.json)。两项滞后/缺口/标签时点检查通过，Notebook执行与NAS回读通过；大预测、模型与损失序列仅在Recorder。
